# Swin Transformer on CIFAR-10

这个 Notebook 从 `CIFAR-10` 数据集加载开始，完整展示 `Resize(224x224) -> Swin Transformer` 的训练与结构分析流程。

内容包括：
- 数据集预处理与可视化
- `DataLoader` 构建
- Patch Partition、Window Attention、Shifted Window Attention 实现
- Patch Merging 与层次化特征图分析
- Token 尺寸变化分析（四个 Stage）
- W-MSA / SW-MSA 注意力机制解读
- 训练、验证与预测展示
- Swin Transformer 与 ViT 的核心差异说明

## 1. 环境准备

如果本地环境尚未安装依赖，可以先执行：

```bash
pip install torch torchvision matplotlib
```

In [ ]:
# 数学工具用于可视化排版等辅助逻辑
import math
# dataclass 用于统一管理实验配置
from dataclasses import dataclass, field
from typing import Tuple

# matplotlib 用于样本和训练曲线可视化
import matplotlib.pyplot as plt
# PyTorch 核心模块
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
# torchvision 提供常见数据集与图像预处理工具
from torchvision import datasets, transforms

# 设置绘图风格，便于 Notebook 展示
plt.style.use('seaborn-v0_8')
# 固定随机种子，便于复现
torch.manual_seed(42)

# 优先使用 GPU，没有则退回 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 数据下载与缓存目录
    data_root: str = './data'
    # 将 CIFAR-10 从 32x32 拉伸到 224x224
    image_size: int = 224
    # Swin 使用 4x4 patch（比 ViT 的 16x16 更细粒度）
    patch_size: int = 4
    # 每个 batch 的样本数
    batch_size: int = 64
    # DataLoader 并行加载进程数
    num_workers: int = 2
    # 学习率
    lr: float = 1e-3
    # 训练轮数
    epochs: int = 5
    # 第一阶段 token embedding 维度
    embed_dim: int = 96
    # 四个 Stage 各有多少 Transformer Block（每两个 block 构成一组 W-MSA+SW-MSA）
    depths: Tuple[int, ...] = field(default_factory=lambda: (2, 2, 6, 2))
    # 四个 Stage 各自的注意力头数（随 dim 翻倍同步增加）
    num_heads: Tuple[int, ...] = field(default_factory=lambda: (3, 6, 12, 24))
    # 局部窗口大小，224/4=56，56 能被 7 整除
    window_size: int = 7
    # MLP 中间层相对于 embed_dim 的放大倍数
    mlp_ratio: float = 4.0
    # Dropout 概率
    dropout: float = 0.1


cfg = Config()
cfg

## 2. 加载 CIFAR-10 并拉伸到 224x224

Swin Transformer 使用 `4x4` patch partition，输入 `224x224` 后得到 `56x56` 的 token 网格。
56 能被 window_size=7 整除，这是参数搭配的关键约束。

In [ ]:
# CIFAR-10 常用归一化参数
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std  = (0.2470, 0.2435, 0.2616)

# 训练集：尺寸拉伸、随机翻转、张量化、归一化
train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 测试集不做随机增强，只保留基础预处理
test_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 本地没有数据时自动下载
train_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=False,
    download=True,
    transform=test_transform,
)

# 类别名称
classes = train_dataset.classes
classes

In [ ]:
def denormalize(image_tensor, mean, std):
    # 反归一化，便于可视化显示
    mean = torch.tensor(mean).view(3, 1, 1)
    std  = torch.tensor(std).view(3, 1, 1)
    return image_tensor * std + mean


# 展示若干样本，确认输入图像形式
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flatten(), range(8)):
    image, label = train_dataset[idx]
    image = denormalize(image, cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(image)
    ax.set_title(classes[label])
    ax.axis('off')

plt.suptitle('CIFAR-10 samples resized to 224x224', fontsize=16)
plt.tight_layout()
plt.show()

## 3. 构建 DataLoader

In [ ]:
# 训练集打乱顺序，减少模型对样本顺序的依赖
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

# 测试集保持固定顺序即可
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

# 检查一个 batch 的维度
images, labels = next(iter(train_loader))
print('batch image shape:', images.shape)
print('batch label shape:', labels.shape)

## 4. Swin Transformer 实现

Swin Transformer（Shifted Window Transformer）相比 ViT 做了两个关键改进：

1. **局部窗口注意力（W-MSA）**：不再对全部 token 做全局 self-attention，而是只在 `7x7` 的局部窗口内做注意力，计算复杂度从 $O(N^2)$ 降到 $O(N \cdot M^2)$。
2. **移位窗口注意力（SW-MSA）**：交替使用正常窗口和位移半个窗口大小的划分方式，让相邻窗口之间的 token 也能间接交流信息。
3. **层次化结构（Patch Merging）**：类似 CNN 的下采样机制，每个 Stage 结束后把相邻 2×2 token 合并，分辨率减半、通道翻倍，最终产生多尺度特征表示。

In [ ]:
def window_partition(x, window_size):
    """
    把 (B, H, W, C) 形状的特征图划分成不重叠的局部窗口。
    返回形状：(num_windows * B, window_size, window_size, C)
    """
    B, H, W, C = x.shape
    # 先把 H、W 分别拆成 (num_windows_h, window_size) 和 (num_windows_w, window_size)
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    # 调整维度顺序，让同一窗口的 token 连续存储
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    windows = windows.view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    """
    window_partition 的逆操作：把所有局部窗口拼回完整特征图。
    返回形状：(B, H, W, C)
    """
    # 根据总窗口数推算 batch size
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x

In [ ]:
class WindowAttention(nn.Module):
    """
    窗口内的 Multi-head Self-Attention，同时支持 W-MSA 和 SW-MSA。
    SW-MSA 通过传入 mask 屏蔽跨窗口的无效注意力权重。
    相对位置偏置（Relative Position Bias）让模型感知窗口内的空间关系。
    """
    def __init__(self, dim, window_size, num_heads, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim         = dim
        self.window_size = window_size  # 窗口边长 M
        self.num_heads   = num_heads
        head_dim         = dim // num_heads
        self.scale       = head_dim ** -0.5

        # 相对位置偏置表：每对位置关系对应一组可学习偏置
        # 窗口内 H 方向的相对距离范围 [-(M-1), M-1]，共 2M-1 个取值
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads)
        )
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

        # 预先计算窗口内所有 token 对的相对位置索引
        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        # coords: (2, M, M) → coords_flatten: (2, M*M)
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing='ij'))
        coords_flatten = torch.flatten(coords, 1)
        # relative_coords: (2, M*M, M*M)，每对 token 的行列相对差
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        # 偏移到非负，再把行偏移乘以 (2M-1) 转成一维索引
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)  # (M*M, M*M)
        self.register_buffer('relative_position_index', relative_position_index)

        self.qkv       = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj      = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.softmax   = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        # x: (num_windows * B, M*M, C)
        B_, N, C = x.shape
        # 把 qkv 一次性算出来再拆分，减少 Linear 调用
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B_, heads, N, head_dim)
        q, k, v = qkv.unbind(0)

        # 缩放点积注意力
        attn = (q * self.scale) @ k.transpose(-2, -1)  # (B_, heads, N, N)

        # 加入相对位置偏置
        rel_pos_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(self.window_size ** 2, self.window_size ** 2, -1)
        rel_pos_bias = rel_pos_bias.permute(2, 0, 1).contiguous()  # (heads, N, N)
        attn = attn + rel_pos_bias.unsqueeze(0)

        # SW-MSA：用大负数屏蔽跨窗口的非法注意力
        if mask is not None:
            nW  = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N)
            attn = attn + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.softmax(attn)
        attn = self.attn_drop(attn)

        # 把注意力权重应用到 v 上，再合并多头
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [ ]:
class SwinTransformerBlock(nn.Module):
    """
    单个 Swin Transformer Block，内部做一次 W-MSA 或 SW-MSA，接 MLP。
    shift_size=0 时为 W-MSA，shift_size=window_size//2 时为 SW-MSA。
    两类 Block 在 BasicLayer 中交替堆叠。
    """
    def __init__(self, dim, input_resolution, num_heads, window_size=7,
                 shift_size=0, mlp_ratio=4., dropout=0.):
        super().__init__()
        self.dim              = dim
        self.input_resolution = input_resolution  # (H, W)
        self.window_size      = window_size
        self.shift_size       = shift_size

        # 如果特征图本身就比窗口小，退化为全局注意力
        if min(input_resolution) <= window_size:
            self.shift_size  = 0
            self.window_size = min(input_resolution)

        self.norm1 = nn.LayerNorm(dim)
        self.attn  = WindowAttention(
            dim=dim,
            window_size=self.window_size,
            num_heads=num_heads,
            proj_drop=dropout,
        )
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, dim),
            nn.Dropout(dropout),
        )

        # 预先计算 SW-MSA 的注意力掩码，避免每次 forward 重复计算
        if self.shift_size > 0:
            H, W       = self.input_resolution
            img_mask   = torch.zeros(1, H, W, 1)
            # 把特征图划分成 9 个区域，每个区域内的 token 属于同一组
            h_slices   = (slice(0, -self.window_size),
                          slice(-self.window_size, -self.shift_size),
                          slice(-self.shift_size, None))
            w_slices   = (slice(0, -self.window_size),
                          slice(-self.window_size, -self.shift_size),
                          slice(-self.shift_size, None))
            cnt = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = cnt
                    cnt += 1
            # 划分窗口，同一窗口内来自不同区域的 token 对设置为 -100
            mask_windows  = window_partition(img_mask, self.window_size)
            mask_windows  = mask_windows.view(-1, self.window_size ** 2)
            attn_mask     = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask     = attn_mask.masked_fill(attn_mask != 0, -100.0)
            attn_mask     = attn_mask.masked_fill(attn_mask == 0,   0.0)
        else:
            attn_mask = None

        self.register_buffer('attn_mask', attn_mask)

    def forward(self, x):
        H, W    = self.input_resolution
        B, L, C = x.shape  # L = H * W

        shortcut = x
        x = self.norm1(x).view(B, H, W, C)

        # SW-MSA：循环移位，把边缘区域滚到中间，让窗口能跨越原始边界
        if self.shift_size > 0:
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))

        # 窗口划分 → 局部注意力 → 还原
        x_windows  = window_partition(x, self.window_size)
        x_windows  = x_windows.view(-1, self.window_size ** 2, C)
        attn_out   = self.attn(x_windows, mask=self.attn_mask)
        attn_out   = attn_out.view(-1, self.window_size, self.window_size, C)
        x          = window_reverse(attn_out, self.window_size, H, W)

        # 反向循环移位，把 token 恢复到原始位置
        if self.shift_size > 0:
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))

        x = x.view(B, L, C)
        # 残差连接
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class PatchMerging(nn.Module):
    """
    层次化下采样：把 2x2 相邻 token 拼接后通过线性层映射，
    空间分辨率减半、通道数翻倍，类似 CNN 中的 stride-2 卷积。
    """
    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution  # (H, W)
        self.dim              = dim
        # 4*dim → 2*dim，将拼接后的 4 份特征压缩，避免通道爆炸
        self.norm      = nn.LayerNorm(4 * dim)
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)

    def forward(self, x):
        H, W    = self.input_resolution
        B, L, C = x.shape
        x       = x.view(B, H, W, C)

        # 以步幅 2 采样 4 个偏移位置，模拟 2x2 邻域内的 token 合并
        x0 = x[:, 0::2, 0::2, :]  # (B, H/2, W/2, C)
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]

        # 通道维度拼接 → (B, H/2, W/2, 4C) → 展平序列
        x = torch.cat([x0, x1, x2, x3], dim=-1).view(B, -1, 4 * C)
        x = self.norm(x)
        x = self.reduction(x)  # (B, H/2 * W/2, 2C)
        return x

In [ ]:
class BasicLayer(nn.Module):
    """
    Swin Transformer 的一个 Stage：若干 SwinTransformerBlock + 可选的 PatchMerging。
    同一 Stage 内相邻 Block 交替使用 W-MSA（shift_size=0）和 SW-MSA（shift_size=M//2）。
    """
    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., dropout=0., downsample=None):
        super().__init__()
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim,
                input_resolution=input_resolution,
                num_heads=num_heads,
                window_size=window_size,
                # 偶数层用正常窗口，奇数层用移位窗口
                shift_size=0 if (i % 2 == 0) else window_size // 2,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
            )
            for i in range(depth)
        ])
        # 最后一个 Stage 不做下采样
        self.downsample = downsample

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x

In [ ]:
class SwinTransformerCIFAR10(nn.Module):
    """
    教学版 Swin-T（Tiny）：
    - 4x4 patch partition → 56x56 token grid
    - 4 个 Stage，每 Stage 结束后（除最后一个）做 Patch Merging
    - Global Average Pooling → 分类头
    """
    def __init__(
        self,
        img_size=224,
        patch_size=4,
        num_classes=10,
        embed_dim=96,
        depths=(2, 2, 6, 2),
        num_heads=(3, 6, 12, 24),
        window_size=7,
        mlp_ratio=4.,
        dropout=0.1,
    ):
        super().__init__()
        self.num_stages = len(depths)
        self.embed_dim  = embed_dim

        # Patch Partition + Linear Embedding：用步幅等于 patch_size 的卷积实现
        self.patch_embed = nn.Conv2d(
            3, embed_dim, kernel_size=patch_size, stride=patch_size
        )
        self.norm_embed  = nn.LayerNorm(embed_dim)
        self.pos_drop    = nn.Dropout(dropout)

        # patch 后的空间分辨率，224//4 = 56
        patches_h = img_size // patch_size
        patches_w = img_size // patch_size

        # 逐 Stage 构建 BasicLayer
        self.layers = nn.ModuleList()
        for i in range(self.num_stages):
            # 每经过一次 PatchMerging，空间分辨率减半
            stage_h   = patches_h // (2 ** i)
            stage_w   = patches_w // (2 ** i)
            stage_dim = int(embed_dim * 2 ** i)

            # 最后一个 Stage 不需要下采样
            downsample = None if (i == self.num_stages - 1) else PatchMerging(
                input_resolution=(stage_h, stage_w),
                dim=stage_dim,
            )

            layer = BasicLayer(
                dim=stage_dim,
                input_resolution=(stage_h, stage_w),
                depth=depths[i],
                num_heads=num_heads[i],
                window_size=window_size,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
                downsample=downsample,
            )
            self.layers.append(layer)

        # 最终输出维度 = embed_dim * 2^(num_stages-1)
        final_dim   = int(embed_dim * 2 ** (self.num_stages - 1))
        self.norm   = nn.LayerNorm(final_dim)
        # 把序列维度上的所有 token 平均成一个向量
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.head    = nn.Linear(final_dim, num_classes)

    def forward(self, x):
        # (B, 3, 224, 224) → (B, embed_dim, 56, 56)
        x = self.patch_embed(x)
        B, C, H, W = x.shape
        # 展平并转置：(B, H*W, C)
        x = x.flatten(2).transpose(1, 2)
        x = self.norm_embed(x)
        x = self.pos_drop(x)

        # 依次经过 4 个 Stage
        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)
        # (B, seq_len, C) → avgpool → (B, C, 1) → squeeze → (B, C)
        x = self.avgpool(x.transpose(1, 2)).flatten(1)
        x = self.head(x)
        return x


model = SwinTransformerCIFAR10(
    img_size=cfg.image_size,
    patch_size=cfg.patch_size,
    num_classes=10,
    embed_dim=cfg.embed_dim,
    depths=cfg.depths,
    num_heads=cfg.num_heads,
    window_size=cfg.window_size,
    mlp_ratio=cfg.mlp_ratio,
    dropout=cfg.dropout,
).to(device)
model

## 5. 层次化 Token 尺寸变化分析

Swin Transformer 的一个核心优势是能产生**层次化特征图**，与 CNN 相似：

| Stage | 输入分辨率 | Token 数 | 维度 |
|-------|-----------|---------|------|
| Patch Embed | 224×224 | 56×56 | 96 |
| Stage 1 + Merging | 56×56 | 28×28 | 192 |
| Stage 2 + Merging | 28×28 | 14×14 | 384 |
| Stage 3 + Merging | 14×14 | 7×7 | 768 |
| Stage 4（无 Merging） | 7×7 | 7×7 | 768 |
| Global AvgPool | — | 1 | 768 |

In [ ]:
@torch.no_grad()
def inspect_stage_shapes(model, input_shape=(1, 3, 224, 224)):
    # 构造假输入，只用于追踪各阶段形状
    model_cpu = model.cpu()
    x = torch.randn(input_shape)
    print(f'input            -> {tuple(x.shape)}')

    # Patch Partition + Embedding
    x = model_cpu.patch_embed(x)
    B, C, H, W = x.shape
    x = x.flatten(2).transpose(1, 2)
    x = model_cpu.norm_embed(x)
    print(f'patch_embed      -> {tuple(x.shape)}  (B, 56*56, 96)')

    # 逐 Stage 追踪
    patches_h = cfg.image_size // cfg.patch_size
    patches_w = cfg.image_size // cfg.patch_size
    for i, layer in enumerate(model_cpu.layers):
        x = layer(x)
        stage_h = patches_h // (2 ** min(i + 1, model_cpu.num_stages - 1))
        stage_w = patches_w // (2 ** min(i + 1, model_cpu.num_stages - 1))
        label   = f'Stage {i+1} output'
        print(f'{label:<16} -> {tuple(x.shape)}')

    x = model_cpu.norm(x)
    x = model_cpu.avgpool(x.transpose(1, 2)).flatten(1)
    print(f'avgpool+flatten  -> {tuple(x.shape)}')

    logits = model_cpu.head(x)
    print(f'head (logits)    -> {tuple(logits.shape)}')


inspect_stage_shapes(model)
model = model.to(device)

## 6. 关键机制解读

### W-MSA：局部窗口注意力
- 把整个特征图切成 `7x7` 的不重叠局部窗口，attention 只在窗口内计算。
- 复杂度从 ViT 的 $O(N^2)$ 降到 $O(\frac{N}{M^2} \cdot M^4) = O(N \cdot M^2)$，N 为 token 总数，M 为窗口大小。

### SW-MSA：移位窗口注意力
- 将窗口划分整体向右下移位 $M/2$，使原本处于不同窗口的 token 有机会进行注意力交互。
- 为避免移位后同一窗口出现来自原始图像不相邻区域的 token 被错误地互相关注，通过循环移位（`torch.roll`）和注意力掩码来维持计算等价性且不引入额外内存开销。

### 相对位置偏置（Relative Position Bias）
- 与 ViT 的绝对位置编码不同，Swin 用可学习的相对位置偏置矩阵加到注意力分数上。
- 窗口内每对 token 的相对位置只有 $(2M-1)^2$ 种，参数量远小于全局方案。

### Patch Merging
- 将 $2 \times 2$ 相邻 token 在通道维度拼接（$4C$），再用线性层映射到 $2C$，分辨率减半、通道翻倍。
- 这使 Swin 能产生类似 ResNet 四阶段的层次化特征，方便接各种 Dense Prediction Head。

### Swin vs ViT
| | ViT | Swin Transformer |
|---|---|---|
| 注意力范围 | 全局（所有 token） | 局部窗口（7×7） |
| 特征图分辨率 | 固定 | 层次化（4 阶段） |
| 计算复杂度 | $O(N^2)$ | $O(N \cdot M^2)$ |
| patch 大小 | 16×16 | 4×4 |
| 适用任务 | 分类 | 分类、检测、分割 |

## 7. 参数量统计

In [ ]:
def count_parameters(model):
    # 只统计可训练参数
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


total_params = count_parameters(model)
print(f'Trainable parameters: {total_params:,}')

## 8. 训练与验证函数

In [ ]:
# 多分类任务常用交叉熵损失
criterion = nn.CrossEntropyLoss()
# 使用 AdamW，Swin 官方训练使用 AdamW + cosine schedule；这里用默认 lr 方便快速实验
optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.05)


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    # 训练模式
    model.train()
    running_loss    = 0.0
    running_correct = 0
    total           = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss    += loss.item() * images.size(0)
        preds            = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total           += labels.size(0)

    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    # 评估模式
    model.eval()
    running_loss    = 0.0
    running_correct = 0
    total           = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss    = criterion(outputs, labels)

        running_loss    += loss.item() * images.size(0)
        preds            = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total           += labels.size(0)

    return running_loss / total, running_correct / total

## 9. 训练主循环

Swin Transformer 和 ViT 类似，对训练资源和训练策略较为敏感。在 GPU 上运行效果更好，CPU 上速度会较慢。
官方使用 AdamW + cosine annealing + 数据增强（RandAugment / Mixup / CutMix）训练 300 epoch。

In [ ]:
# 记录训练和验证指标，便于后面画曲线
history = {
    'train_loss': [],
    'train_acc':  [],
    'val_loss':   [],
    'val_acc':    [],
}

# 每轮先训练，再评估
for epoch in range(cfg.epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = evaluate(model, test_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'Epoch [{epoch + 1}/{cfg.epochs}] '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

In [ ]:
# 绘制损失曲线和准确率曲线
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], label='train loss')
axes[0].plot(epochs_range, history['val_loss'],   label='val loss')
axes[0].set_title('Loss curves')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_range, history['train_acc'], label='train acc')
axes[1].plot(epochs_range, history['val_acc'],   label='val acc')
axes[1].set_title('Accuracy curves')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. 预测结果展示

In [ ]:
@torch.no_grad()
def show_predictions(model, dataloader, class_names, device, num_images=8):
    # 切换到评估模式，保证推理行为稳定
    model.eval()
    # 取一个 batch 做展示
    images, labels = next(iter(dataloader))
    images = images.to(device)
    labels = labels.to(device)

    logits = model(images)
    preds  = logits.argmax(dim=1)

    fig, axes = plt.subplots(2, math.ceil(num_images / 2), figsize=(16, 6))
    axes = axes.flatten()

    for i in range(num_images):
        # 反归一化后再显示图像
        image = denormalize(images[i].cpu(), cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1)
        axes[i].imshow(image)
        axes[i].set_title(f'true: {class_names[labels[i]]}\npred: {class_names[preds[i]]}')
        axes[i].axis('off')

    for i in range(num_images, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()


show_predictions(model, test_loader, classes, device)